# Watch the biped train in parallel (mjlab viser viewer on Colab)

A **separate** companion to `notebooks/train_biped.ipynb` — it does **not** change or replace it.
This notebook embeds mjlab's **viser** web viewer directly in the Colab output as an iframe, so you can
watch **many copies of the robot at once** (the same parallelism that makes mjlab fast) and visually
verify the policy is actually learning to balance/walk.

**How this mirrors the mjlab demo notebook:** the demo runs a viewer that serves a web UI on a local
port, then embeds that port with `google.colab.output.serve_kernel_port_as_iframe`. This notebook does
the same, but points the viewer at **our** task (`Mjlab-Biped-Balance-v0`) instead of the pretrained
humanoid demo.

**Requires a GPU runtime** (Runtime > Change runtime type > GPU) for training to be fast; the viewer
itself will run on CPU too, just slower.

**Two clarifications up front, so the workflow makes sense:**
1. mjlab trains **headless** — thousands of envs step on the GPU with no rendering (that is *why* it is
   fast). You do not watch the literal gradient loop; instead you **replay saved checkpoints** in the
   viewer. The viewer even has a checkpoint dropdown, so as training drops new checkpoints you can reload
   them and watch the policy improve.
2. **Train with thousands of envs** (parallelism = fast learning); **view with ~16 envs** (more than that
   just clutters the browser and slows rendering). These two `num_envs` are deliberately different.


## Use W&B offline

Disables W&B cloud sync. Run this **or** the login cell below — not both.


In [ ]:
!wandb offline


## **Or** login using an API key from your W&B account


In [ ]:
!wandb login


## GPU runtime check


In [ ]:
!nvidia-smi


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: no GPU detected. The viewer still works on CPU, but training will be slow.\n'
          'For real training set Runtime > Change runtime type > GPU.')


## Mount Google Drive (required — checkpoints must persist here)

Same reason as in `train_biped.ipynb`: the Colab local disk is ephemeral. If you train from
*this* notebook's Section 2, checkpoints need to land on Drive to survive a runtime disconnect.
If you already trained from `train_biped.ipynb` in a **separate** Colab session, mount the same
Drive account here too — `DRIVE_LOG_ROOT` below points at the same path that notebook writes to,
so Section 3 will find those checkpoints.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# Must match train_biped.ipynb's DRIVE_LOG_ROOT -- this is where checkpoints are read from
# (Section 3) and, if you train here instead, written to (Section 2).
DRIVE_LOG_ROOT = "/content/drive/MyDrive/biped_training/logs/rsl_rl"
os.makedirs(DRIVE_LOG_ROOT, exist_ok=True)
print("Checkpoints will be read/written under:", DRIVE_LOG_ROOT)


## Install dependencies and verify the upload bundle

Upload and unzip the same `colab_bundle.zip` used by the training notebook (see
`docs/colab_upload_manifest.md`). It must be unpacked into the working directory **before** running
the cells below. `mjlab` bundles `mujoco-warp` and the `viser` viewer, so no extra install is needed
for the viewer.


In [ ]:
# The bundle's requirements-colab.txt must already be present in the working directory.
!pip install -r requirements-colab.txt


In [ ]:
# Verify the upload bundle is complete (same check as the training notebook, plus the
# scripts/ directory this notebook needs for the viewer driver).
import os

expected_paths = [
    'models/mjcf/biped_warp.xml',
    'meshes/stl',
    'mjlab_biped',
    'requirements-colab.txt',
    'scripts/colab_play.py',
    'scripts/colab_train.py',
]
missing = [p for p in expected_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f'Upload bundle is incomplete, missing: {missing}. '
        f'Re-download the latest colab_bundle.zip and unzip it here. '
        f'See docs/colab_upload_manifest.md for the full expected layout.'
    )
print('Bundle looks complete:', sorted(os.listdir('.')))


In [ ]:
import mjlab
import mujoco_warp
import rsl_rl
import importlib.metadata

def _version(pkg_name, module):
    try:
        return module.__version__
    except AttributeError:
        return importlib.metadata.version(pkg_name)

print('mjlab version:', _version('mjlab', mjlab))
print('mujoco_warp version:', _version('mujoco-warp', mujoco_warp))
print('rsl_rl version:', _version('rsl-rl-lib', rsl_rl))


## Register the biped task

Importing `mjlab_biped.mjlab_task` registers `Mjlab-Biped-Balance-v0` into mjlab's task registry
(same as the training notebook). The viewer driver `scripts/colab_play.py` does this import for us in
its own process too — this cell is just a fast sanity check that the task is discoverable.


In [ ]:
import mjlab_biped.mjlab_task as biped_task
from mjlab.tasks.registry import list_tasks

print('Registered task:', biped_task.TASK_ID)
assert biped_task.TASK_ID in list_tasks(), 'Task registration failed'
print('Task is discoverable in the mjlab registry.')


## The viewer helper

`launch_viser(...)` starts mjlab's viser viewer as a **background subprocess** (via
`scripts/colab_play.py`, so the task is registered in that process), pins it to a known port with
viser's `_VISER_PORT_OVERRIDE`, parses the port viser actually bound from its startup log, and returns
it. The next cell embeds that port as an iframe.

- `agent='zero'` / `'random'` — **no checkpoint needed**. Use this to verify parallel rendering *before*
  you have a trained policy. `'zero'` holds the stand-pose target (calmer picture); `'random'` sends
  random actions (obviously 'alive', flails and falls).
- `agent='trained'` — replays a saved checkpoint. Pass `checkpoint_file=...`, or leave it `None` to
  auto-pick the newest `model_*.pt` under `DRIVE_LOG_ROOT` (defined in the Drive-mount section above).


In [ ]:
import subprocess, os, sys, re, time, threading, signal, glob

_VIEWER_PROC = None  # global handle so we can stop/relaunch the viewer

def stop_viser():
    """Terminate any running viser viewer subprocess (frees the port)."""
    global _VIEWER_PROC
    if _VIEWER_PROC is not None and _VIEWER_PROC.poll() is None:
        _VIEWER_PROC.send_signal(signal.SIGINT)
        try:
            _VIEWER_PROC.wait(timeout=5)
        except subprocess.TimeoutExpired:
            _VIEWER_PROC.kill()
    _VIEWER_PROC = None

def find_latest_checkpoint(log_root=None):
    """Newest model_*.pt under log_root (mjlab saves every save_interval iters).
    Defaults to DRIVE_LOG_ROOT (defined in the Drive-mount section above) so
    checkpoints are found on Drive, not the ephemeral local disk."""
    if log_root is None:
        log_root = DRIVE_LOG_ROOT
    ckpts = glob.glob(os.path.join(log_root, '**', 'model_*.pt'), recursive=True)
    return max(ckpts, key=os.path.getmtime) if ckpts else None

def launch_viser(agent='zero', num_envs=16, checkpoint_file=None, port=8080,
                 startup_timeout=240):
    """Launch the mjlab viser viewer as a background subprocess; return the bound port."""
    global _VIEWER_PROC
    stop_viser()  # kill any previous viewer holding the port

    cmd = [sys.executable, 'scripts/colab_play.py', biped_task.TASK_ID,
           '--agent', agent, '--num-envs', str(num_envs), '--viewer', 'viser']
    if agent == 'trained':
        if checkpoint_file is None:
            checkpoint_file = find_latest_checkpoint()
        if not checkpoint_file or not os.path.exists(checkpoint_file):
            raise FileNotFoundError(
                "agent='trained' needs a checkpoint. Train first (the training cell below, "
                'or notebooks/train_biped.ipynb), or pass checkpoint_file=... explicitly.')
        cmd += ['--checkpoint-file', checkpoint_file]
        print('Using checkpoint:', checkpoint_file)

    env = dict(os.environ)
    env['_VISER_PORT_OVERRIDE'] = str(port)  # pin viser to this port
    env['MJLAB_WARP_QUIET'] = '1'            # quiet warp kernel-compile logs

    _VIEWER_PROC = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=env)

    detected, start = None, time.time()
    for line in iter(_VIEWER_PROC.stdout.readline, ''):
        print(line, end='')
        m = (re.search(r'listening \*:(\d+)', line)
             or re.search(r'http://(?:localhost|0\.0\.0\.0|127\.0\.0\.1):(\d+)', line))
        if m:
            detected = int(m.group(1)); break
        if _VIEWER_PROC.poll() is not None:
            raise RuntimeError('Viewer subprocess exited before viser started '
                               '(see the traceback above).')
        if time.time() - start > startup_timeout:
            raise TimeoutError(f'viser did not start within {startup_timeout}s.')

    # Drain remaining stdout in a daemon thread so the pipe buffer never fills
    # (a full pipe would block the subprocess and freeze the simulation).
    threading.Thread(target=lambda p: [None for _ in iter(p.stdout.readline, '')],
                     args=(_VIEWER_PROC,), daemon=True).start()

    print(f"\n{'='*60}\n[OK] viser running on port {detected} - embedding below.\n{'='*60}")
    return detected


## 1. Watch parallel envs BEFORE training (verifies rendering + parallelism)

No checkpoint required. This launches 16 copies of the robot holding the stand-pose target. With no
balance controller yet they will drift and topple — that is expected; the point is to confirm the
parallel envs render live in the iframe below.


In [ ]:
port = launch_viser(agent='zero', num_envs=16)   # try agent='random' for a livelier picture


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(port, height=650)


## 2. Train (headless, GPU, thousands of parallel envs)

This is where mjlab's parallelism pays off: run **thousands** of envs on the GPU with no rendering.
Checkpoints are written under `DRIVE_LOG_ROOT` (on Drive, from the Mount section above) every
`save_interval` (50) iterations. Start small to confirm it learns, then raise `--agent.max-iterations`.
`--agent.wandb-project` sets which W&B project the run logs to (mjlab defaults to the generic
`"mjlab"` project if you don't set this).

> Stop this cell whenever you have a checkpoint or two — you do not need to let it finish to view
> progress. Section 3 will pick up the newest checkpoint automatically.


In [ ]:
# Train with MANY envs (parallelism = fast). Adjust num-envs to your GPU's memory.
!python scripts/colab_train.py Mjlab-Biped-Balance-v0 \
    --env.scene.num-envs 4096 --agent.max-iterations 300 \
    --log-root {DRIVE_LOG_ROOT} \
    --agent.wandb-project biped-balance \
    --agent.experiment-name biped-run1


## 3. Watch the TRAINED policy in parallel (verify it is learning)

Replays the newest checkpoint in 16 parallel envs. Re-run these two cells after more training to watch
the policy improve. (Inside the viser UI there is also a checkpoint dropdown that hot-swaps checkpoints
without relaunching — handy while training keeps dropping new ones.)


In [ ]:
# checkpoint_file=None auto-picks the newest model_*.pt under DRIVE_LOG_ROOT.
# Pass an explicit path to view a specific checkpoint.
port = launch_viser(agent='trained', num_envs=16, checkpoint_file=None)


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(port, height=650)


## What to look for (verifying the training process)

- **Early checkpoints:** robots topple quickly, much like the `agent='zero'` view.
- **As training progresses:** episodes last longer, the torso stays upright, feet stay under the CoM.
  For the balance gate the target is staying upright for the full episode in the large majority of envs.
- **Reward hacking watch-out:** if the robot survives by high-frequency vibration/buzzing rather than
  genuine balance, that is the actuator being exploited — worth catching visually here, not just from
  the reward curve.

**To stop the viewer** (free the port before relaunching): `stop_viser()`.

**Why you cannot watch the literal training loop:** during `train`, envs step on the GPU with rendering
off — that headless batching is exactly what makes thousands of parallel envs affordable. The viewer
replays checkpoints instead, which is the intended mjlab workflow for eyeballing progress.
